# Exponential integrators in NGSolve — Part 2: computing $e^{-tA}v$ for a large sparse FE operator

Part 1 introduced exponential Euler and used shift-and-invert Lanczos to apply $e^{-\Delta t A}$, $A = M^{-1}K$. This notebook is a survey of the practical alternatives and of their cost, always for the finite element setting where

* a product with $A$ costs one product with $K$ **plus one solve with the mass matrix** $M$,
* a solve with $(I + \gamma A) = M^{-1}(M + \gamma K)$ costs one sparse back-substitution with a factorisation of $M + \gamma K$ (the implicit Euler matrix),
* a solve with $(zI + A)$ for complex $z$ costs one *complex* back-substitution.

We compare

| method | what it needs | iterations for tolerance $\varepsilon$ | mesh dependent? |
|---|---|---|---|
| polynomial Lanczos (Saad, Hochbruck–Lubich) | products with $A$ | $\approx \sqrt{\Delta t\lambda_{\max}} + O(\log 1/\varepsilon)$ | yes |
| truncated Taylor, `scipy.sparse.linalg.expm_multiply` (Al-Mohy–Higham) | products with $A$ | $\propto \Delta t\lambda_{\max}$ | yes |
| Chebyshev expansion (Druskin–Knizhnerman, Bergamaschi–Vianello) | products with $A$ + a bound for $\lambda_{\max}$ | $\approx \Delta t\lambda_{\max}/2 + O(\log 1/\varepsilon)$ | yes |
| **shift-and-invert Lanczos** (van den Eshof–Hochbruck) | one factorisation of $M+\gamma K$ | 10–30 | **no** |
| **contour integral / rational approximation** (Trefethen–Weideman–Schmelzer) | $N/2$ complex factorisations of $z_kM + K$ | $N = 16$–$32$ | **no** |

The helpers of Part 1 (and a few more) are collected in `expint_tools.py` one directory up.

In [ ]:
import sys; sys.path.insert(0, "..")
from ngsolve import *
from ngsolve.webgui import Draw
from ngsolve.krylovspace import CGSolver
import numpy as np, scipy.linalg as sla, scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from time import time as walltime
from expint_tools import phi, krylov_funm, KrylovExp, ContourExp, chebyshev_exp, lanczos_spectral_bound, DenseReference
ngsglobals.msg_level = 0

## Setup: the heat equation on two meshes

Same model problem as in Part 1 (unit square, Dirichlet everywhere, order 3, rough initial datum). On the coarse mesh the dense generalised eigenproblem gives us the exact $e^{-tA}v$; on the fine mesh we use shift-and-invert Lanczos with a very tight tolerance as reference and cross-check it against the contour integral.

In [ ]:
def setup(maxh):
    mesh = Mesh(unit_square.GenerateMesh(maxh=maxh))
    fes = H1(mesh, order=3, dirichlet=".*")
    u, v = fes.TnT()
    K = BilinearForm(grad(u)*grad(v)*dx, symmetric=False).Assemble().mat
    M = BilinearForm(u*v*dx, symmetric=False).Assemble().mat
    fesc = H1(mesh, order=3, dirichlet=".*", complex=True)               # for complex shifts
    uc, vc = fesc.TnT()
    Kc = BilinearForm(grad(uc)*grad(vc)*dx, symmetric=False).Assemble().mat
    Mc = BilinearForm(uc*vc*dx, symmetric=False).Assemble().mat
    gfu = GridFunction(fes)
    gfu.Set(exp(-100*((x-0.3)**2 + (y-0.6)**2)) + IfPos((x-0.5)*(0.8-x), 1, 0)*IfPos((y-0.2)*(0.5-y), 1, 0))
    gfu.vec.data = Projector(fes.FreeDofs(), True) * gfu.vec
    invM = M.Inverse(fes.FreeDofs(), inverse="sparsecholesky")
    tmp = M.CreateColVector()
    def applyA(z):                                                       # z -> M^{-1} K z
        out = M.CreateColVector(); tmp.data = K*z; out.data = invM*tmp; return out
    return dict(mesh=mesh, fes=fes, K=K, M=M, Kc=Kc, Mc=Mc, freedofs=fes.FreeDofs(), freedofsc=fesc.FreeDofs(),
                u0=gfu.vec, invM=invM, applyA=applyA, ndof=fes.ndof)

coarse, fine = setup(0.1), setup(0.04)
ref = DenseReference(coarse["K"], coarse["M"], coarse["freedofs"])
for name, P in [("coarse", coarse), ("fine", fine)]:
    P["lam_max"] = lanczos_spectral_bound(P["applyA"], P["M"], P["u0"], m=40) / 1.05
    print(f"{name} mesh: ndof = {P['ndof']:5d},   lambda_max ~ {P['lam_max']:.3e}")
print(f"coarse mesh, dense: lambda_max = {ref.lam[-1]:.3e}")

## 1. Polynomial Lanczos

$e^{-\Delta t A}v \approx \beta V_m e^{-\Delta t T_m} e_1$ with $V_m$ an $M$-orthonormal basis of $\mathcal K_m(A, v)$. Hochbruck and Lubich (1997) proved for symmetric $A$ with spectrum in $[0, \rho/\Delta t]$ — i.e. $\rho = \Delta t\lambda_{\max}$ — the bound

$$
\|e^{-\Delta t A}v - \beta V_m e^{-\Delta t T_m}e_1\|
\;\le\; 10\, e^{-m^2/(5\rho)} \|v\| \quad (\sqrt{4\rho} \le m \le 2\rho), \qquad
\le\; 10\,\rho^{-1} e^{-\rho}\Big(\frac{e\rho}{m}\Big)^m \|v\| \quad (m \ge 2\rho).
$$

Nothing happens before $m \approx \sqrt{\rho}$, then convergence is superlinear. Since $\lambda_{\max} \sim h^{-2}$, the number of iterations grows like $\sqrt{\Delta t}/h$.

In [ ]:
dts = [1e-3, 1e-2, 1e-1]
plt.figure(figsize=(6, 4))
for dt in dts:
    ex = ref.apply(lambda l: np.exp(-dt*l), ref.to_np(coarse["u0"]))
    ms = list(range(5, 160, 5)); errs = []
    for m in ms:
        errs.append(ref.relerr(krylov_funm(coarse["applyA"], coarse["M"], coarse["u0"], lambda th: np.exp(-dt*th), maxm=m, tol=0), ex))
    rho = dt*ref.lam[-1]
    l, = plt.semilogy(ms, errs, "o-", ms=3, label=rf"$\Delta t$ = {dt:g},  $\rho = \Delta t\lambda_{{max}}$ = {rho:.0f}")
    plt.axvline(np.sqrt(rho), color=l.get_color(), ls=":")
plt.ylim(1e-14, 10); plt.xlabel("Krylov dimension m"); plt.ylabel("rel. error"); plt.grid(); plt.legend()
plt.title(r"polynomial Lanczos, dotted: $m = \sqrt{\rho}$"); plt.show()

The number of iterations to reach a given tolerance on the two meshes ($h$ is halved: $\lambda_{\max}$ grows by 4, $m$ by 2):

In [ ]:
print(f"{'dt':>8s} {'coarse m':>10s} {'fine m':>8s}")
for dt in dts:
    ms = []
    for P in (coarse, fine):
        E = KrylovExp(P["M"], P["K"], dt, P["freedofs"], poly=True, tol=1e-8, maxm=400)
        (E * P["u0"]).Evaluate(); ms.append(E.info["m"])
    print(f"{dt:8g} {ms[0]:10d} {ms[1]:8d}")

### Error estimation

The Lanczos relation $A V_m = V_m T_m + t_{m+1,m} v_{m+1} e_m^T$ gives an expression for the residual of the Krylov approximation $y_m(t) = \beta V_m e^{-tT_m} e_1$ as a solution of the ODE $y' = -Ay$:

$$
r_m(t) := -A y_m(t) - y_m'(t) = -\beta\, t_{m+1,m}\, \big(e_m^T e^{-tT_m}e_1\big)\, v_{m+1},
\qquad \|r_m(t)\|_M = \beta\, t_{m+1,m}\, |e_m^T e^{-tT_m}e_1| .
$$

This is the classical a-posteriori estimate of Saad (1992); it is also the first term of an error expansion and is usually within a factor of 10 of the true error. Our `krylov_funm` uses the simpler difference between consecutive approximations. Both are compared here for $\Delta t = 0.01$.

In [ ]:
dt = 1e-2
ex = ref.apply(lambda l: np.exp(-dt*l), ref.to_np(coarse["u0"]))
M, v, applyA = coarse["M"], coarse["u0"], coarse["applyA"]
beta = sqrt(InnerProduct(v, M*v)); V = MultiVector(v, 0); V.Append((1/beta)*v)
maxm = 100; T = np.zeros((maxm+1, maxm)); saad, diffs, true = [], [], []; Eold = np.zeros(0)
for j in range(maxm):
    h = np.array(V.AppendOrthogonalize(applyA(V[j]), ipmat=M)).ravel(); T[:j+2, j] = h[:j+2]
    th, Q = sla.eigh(T[:j+1, :j+1]); E = beta * (Q @ (np.exp(-dt*th) * Q[0, :]))
    saad.append(beta*abs(h[j+1]*E[-1]) / beta)                                   # Saad residual (relative)
    diffs.append(np.linalg.norm(E - np.append(Eold, np.zeros(j+1-len(Eold)))) / beta); Eold = E
    true.append(ref.relerr((V[0:j+1]*Vector(E)).Evaluate(), ex))
plt.figure(figsize=(6, 4)); plt.semilogy(range(1, maxm+1), true, "k-", label="true error")
plt.semilogy(range(1, maxm+1), saad, "--", label="Saad residual $\\beta\\, t_{m+1,m} |e_m^T e^{-\\Delta t T_m} e_1|$")
plt.semilogy(range(1, maxm+1), diffs, ":", label="difference of consecutive iterates")
plt.ylim(1e-14, 1e3); plt.xlabel("m"); plt.grid(); plt.legend(); plt.show()

## 2. Truncated Taylor series: `scipy.sparse.linalg.expm_multiply`

Al-Mohy and Higham (2011) combine scaling, $e^{-tA} = (e^{-tA/s})^s$, with a truncated Taylor series of degree $\le 55$ for each factor, choosing $s$ from 1-norm estimates of $A$ so that the truncation error is at unit roundoff. It is matrix-free and robust (also for non-normal $A$), but the total number of products with $A$ is proportional to $\Delta t\,\|A\|$, i.e. linear in $\rho$ instead of $\sqrt\rho$ for Lanczos. `scipy` implements exactly this algorithm and accepts a `LinearOperator`; for the 1-norm estimation it also needs the transpose $A^T = K M^{-1}$.

In [ ]:
def expm_multiply_FE(P, dt, count=[0]):
    M, K, invM, n = P["M"], P["K"], P["invM"], P["ndof"]
    tmpK, tmpM, out = M.CreateColVector(), M.CreateColVector(), M.CreateColVector()
    count[0] = 0
    def matvec(z):                                        # M^{-1} K z  (numpy in, numpy out)
        count[0] += 1; tmpK.FV().NumPy()[:] = np.asarray(z).ravel(); tmpM.data = K*tmpK; out.data = invM*tmpM
        return out.FV().NumPy().copy()
    def rmatvec(z):                                       # K M^{-1} z
        count[0] += 1; tmpK.FV().NumPy()[:] = np.asarray(z).ravel(); tmpM.data = invM*tmpK; out.data = K*tmpM
        return out.FV().NumPy().copy()
    Aop = spla.LinearOperator((n, n), matvec=matvec, rmatvec=rmatvec, dtype=float)
    res = P["u0"].CreateVector()
    res.FV().NumPy()[:] = spla.expm_multiply(-dt*Aop, P["u0"].FV().NumPy(), traceA=-dt*float(np.sum(ref.lam)) if P is coarse else None)
    return res, count[0]

for dt in dts:
    ex = ref.apply(lambda l: np.exp(-dt*l), ref.to_np(coarse["u0"]))
    t0 = walltime(); res, nmv = expm_multiply_FE(coarse, dt); t1 = walltime() - t0
    print(f"expm_multiply, coarse mesh, dt = {dt:g}:  {nmv:5d} products with A,  rel. error {ref.relerr(res, ex):.1e},  {t1:.2f} s")

## 3. Chebyshev expansion

For symmetric $A$ with spectrum in $[0, L]$ one can expand $e^{-\Delta t\lambda}$ in Chebyshev polynomials on that interval. With $s = (2\lambda - L)/L \in [-1, 1]$ and $a = \Delta t L/2$,

$$
e^{-\Delta t\lambda} = e^{-a} e^{-a s} = \sum_{k \ge 0} c_k T_k(s), \qquad c_k = 2(-1)^k e^{-a} I_k(a)
$$

($I_k$ modified Bessel functions, $c_0$ halved), and the three-term recurrence $T_{k+1}(S)v = 2 S\,T_k(S)v - T_{k-1}(S)v$ with $S = (2A - LI)/L$ needs only three vectors of storage and no inner products — attractive on GPUs and for very large problems where storing a Krylov basis is impossible. The coefficients decay only once $k > a \approx \rho/2$, so the degree grows *linearly* in $\rho$ (like the Taylor method) and the spectral bound $L \ge \lambda_{\max}$ must be safe: if $L < \lambda_{\max}$ the recurrence blows up. We get $L$ from a few Lanczos steps (largest Ritz value plus its residual bound, times a safety factor), see `lanczos_spectral_bound`.

In [ ]:
L = lanczos_spectral_bound(coarse["applyA"], coarse["M"], coarse["u0"], m=30)
print(f"spectral bound L = {L:.4e}   (true lambda_max = {ref.lam[-1]:.4e})")
plt.figure(figsize=(6, 4))
for dt in dts:
    ex = ref.apply(lambda l: np.exp(-dt*l), ref.to_np(coarse["u0"]))
    a = dt*L/2; degs = np.unique(np.linspace(5, 2*a + 120, 20).astype(int)); errs = []
    for n in degs:
        errs.append(ref.relerr(chebyshev_exp(coarse["applyA"], coarse["u0"], dt, L, int(n)), ex))
    l, = plt.semilogy(degs, errs, "o-", ms=3, label=rf"$\Delta t$ = {dt:g},  $a = \Delta t L/2$ = {a:.0f}")
    plt.axvline(a, color=l.get_color(), ls=":")
plt.ylim(1e-15, 10); plt.xlabel("degree n"); plt.ylabel("rel. error"); plt.grid(); plt.legend(); plt.title("Chebyshev expansion, dotted: n = a"); plt.show()

## 4. Shift-and-invert Lanczos

The Krylov space is built with $B = (M + \gamma K)^{-1}M = (I + \gamma A)^{-1}$; the small matrix $\tilde T_m$ approximates $B$ and we evaluate $g(\tilde T_m)$ with $g(\theta) = e^{-\Delta t(1/\theta - 1)/\gamma}$. Van den Eshof and Hochbruck (2006) show that the error is bounded by the best approximation error of $e^{-\Delta t\lambda}$ by rational functions with the single pole $-1/\gamma$ of multiplicity $m$, which is **independent of $\lambda_{\max}$**, and give the following guidance for the shift: $\gamma \approx \Delta t/10$ for tolerances around $10^{-6}$, smaller ($\Delta t/15$ to $\Delta t/20$) for tighter ones; asymptotically $\gamma_{\rm opt} \approx \sqrt2\,\Delta t/m$. Too large a $\gamma$ (e.g. $\gamma = \Delta t$) still converges but more slowly. In practice $\gamma$ is fixed once, since changing it means a new factorisation.

In [ ]:
dt = 1e-2
ex = ref.apply(lambda l: np.exp(-dt*l), ref.to_np(coarse["u0"]))
plt.figure(figsize=(6, 4))
for gfac in [0.02, 0.05, 0.1, 0.3, 1.0, 3.0]:
    E = KrylovExp(coarse["M"], coarse["K"], dt, coarse["freedofs"], gamma=gfac*dt, tol=0)
    ms = list(range(2, 41, 2)); errs = []
    for m in ms:
        E.maxm = m; errs.append(ref.relerr((E*coarse["u0"]).Evaluate(), ex))
    plt.semilogy(ms, errs, "o-", ms=3, label=rf"$\gamma = {gfac:g}\,\Delta t$")
plt.ylim(1e-14, 1); plt.xlabel("m"); plt.ylabel("rel. error"); plt.grid(); plt.legend(); plt.title(rf"shift-and-invert Lanczos, $\Delta t$ = {dt:g}"); plt.show()

In [ ]:
print("Krylov dimension for tol = 1e-8 (gamma = dt/10):")
print(f"{'dt':>8s} {'coarse m':>10s} {'fine m':>8s}")
for dt in dts:
    ms = []
    for P in (coarse, fine):
        E = KrylovExp(P["M"], P["K"], dt, P["freedofs"], tol=1e-8)
        (E * P["u0"]).Evaluate(); ms.append(E.info["m"])
    print(f"{dt:8g} {ms[0]:10d} {ms[1]:8d}")

### Inexact inner solves

The solves with $M + \gamma K$ need not be exact: replacing the factorisation by a preconditioned CG iteration with a moderate tolerance still gives a useful approximation (the accuracy is limited by the inner tolerance), which matters in 3D where factorisations are expensive. Here with a Jacobi preconditioner (the right-hand side must be projected onto the free dofs, since CG does not know about the Dirichlet rows). Note that with a loose inner tolerance the outer stopping criterion never triggers — the iterates keep changing at the level of the inner error — so `maxm` becomes the effective stop:

In [ ]:
M, K, freedofs, v = coarse["M"], coarse["K"], coarse["freedofs"], coarse["u0"]
dt = 1e-2; gamma = 0.1*dt
ex = ref.apply(lambda l: np.exp(-dt*l), ref.to_np(v))
mstar = M.CreateMatrix(); mstar.AsVector().data = M.AsVector() + gamma*K.AsVector()
jac = mstar.CreateSmoother(freedofs)
Pfree = Projector(freedofs, True)
tmp = M.CreateColVector()
def g_exp(th):
    with np.errstate(divide="ignore"): return np.exp(-dt*np.where(th > 0, (1/th - 1)/gamma, np.inf))
for cgtol in [1e-3, 1e-6, 1e-9, 1e-12]:
    cg = CGSolver(mstar, jac, tol=cgtol, maxiter=500, printrates=False)
    def opB(z):
        out = M.CreateColVector(); tmp.data = M*z; tmp.data = Pfree*tmp; out.data = cg*tmp; return out
    info = {}
    yk = krylov_funm(opB, M, v, g_exp, maxm=40, tol=1e-10, info=info)
    print(f"inner CG tol {cgtol:.0e}:  Krylov dim {info['m']:2d},  rel. error {ref.relerr(yk, ex):.1e}")

## 5. Contour integrals (rational approximation with complex shifts)

Cauchy's integral formula for $f(-\Delta t A)$ with a contour $\Gamma$ enclosing the (negative) spectrum of $-A$, i.e. the interval $[-\lambda_{\max}, -\lambda_{\min}]$:

$$
e^{-\Delta t A}v = \frac{1}{2\pi i}\oint_\Gamma e^{\Delta t z}\,(zI + A)^{-1} v\, dz
                 = \frac{1}{2\pi i}\oint_\Gamma e^{\Delta t z}\,(zM + K)^{-1} M v\, dz .
$$

Since $e^{\Delta t z}$ decays for $\mathrm{Re}\,z \to -\infty$, $\Gamma$ can be an open curve around the negative real axis, and the trapezoidal rule on such a curve converges geometrically. Weideman and Trefethen (2007) optimised the parabola

$$
z(\theta) = \frac{N}{\Delta t}\big(0.1309 - 0.1194\,\theta^2 + 0.25\, i\theta\big), \quad \theta \in (-\pi, \pi),
$$

with $N$ midpoint-rule nodes to an error of $O(2.85^{-N})$; by conjugate symmetry only the $N/2$ nodes with $\theta > 0$ are needed for real data. The result is a *rational* approximation $\sum_k w_k (z_kM + K)^{-1}M$ with $N/2$ complex poles, uniformly accurate for all $v$, requiring $N/2$ complex factorisations (once) and $N/2$ back-substitutions per application — all independent of each other, hence trivially parallel, and with no inner products or adaptivity. NGSolve needs the matrices on a complex `FESpace` to form $z_kM + K$.

The class `ContourExp` does this; its `Phi(k, x)` uses the recursion $\varphi_k(-\Delta tA) = -(\Delta t A)^{-1}\big(\varphi_{k-1}(-\Delta tA) - I/(k-1)!\big)$ with one extra factorisation of $K$, because $\varphi_k(\Delta t z)$ decays only like $1/z$ along the contour and the trapezoidal rule would lose its geometric convergence.

In [ ]:
Ns = [4, 8, 12, 16, 20, 24, 28, 32]
plt.figure(figsize=(6, 4))
for dt in dts:
    ex = ref.apply(lambda l: np.exp(-dt*l), ref.to_np(coarse["u0"]))
    errs = [ref.relerr((ContourExp(coarse["Mc"], coarse["Kc"], dt, coarse["freedofsc"], N=N, M=coarse["M"]) * coarse["u0"]).Evaluate(), ex) for N in Ns]
    plt.semilogy(Ns, errs, "o-", label=rf"$\Delta t$ = {dt:g}")
plt.semilogy(Ns, [2.85**(-N) for N in Ns], "k:", label="$2.85^{-N}$")
plt.xlabel("number of contour points N  (N/2 complex solves)"); plt.ylabel("rel. error"); plt.grid(); plt.legend(); plt.title("parabolic contour"); plt.show()

dt = 1e-2
C = ContourExp(coarse["Mc"], coarse["Kc"], dt, coarse["freedofsc"], N=24, M=coarse["M"])
g = (coarse["invM"] * coarse["u0"]).Evaluate()
for k in [1, 2]:
    print(f"phi_{k} via contour (N=24):  rel. error {ref.relerr(C.Phi(k, g), ref.apply(lambda l: phi(k, -dt*l), ref.to_np(g))):.1e}")

The same idea with the *best* rational approximation of $e^{z}$ on $(-\infty, 0]$ (Carathéodory–Fejér, `cf` in Trefethen–Weideman–Schmelzer 2006) needs even fewer poles: the type $(n,n)$ error is $\approx 2\cdot 9.289^{-n}$, i.e. $10^{-12}$ with $n = 12$, $10^{-14}$ with $n = 14$. The `AAA` algorithm (`scipy.interpolate.AAA`) can produce such approximants for arbitrary functions, including the $\varphi_k$.

## 6. The augmented matrix trick for $\varphi$-functions

For a matrix $T$ and vector $b$ (Saad 1992, Sidje 1998, Al-Mohy–Higham 2011)

$$
\exp\begin{pmatrix} T & b \\ 0 & 0\end{pmatrix} = \begin{pmatrix} e^{T} & \varphi_1(T)\, b \\ 0 & 1\end{pmatrix},
$$

so one exponential of a matrix of dimension $m+1$ gives $\varphi_1(T)b$, and more generally a $(m+p)$-dimensional exponential gives $\sum_{k\le p}\varphi_k(T)b_k$. Software like `expokit`, `phipm` or `KIOPS` applies this to the *large* operator $A$ (the augmented matrix is non-symmetric, so Arnoldi is used). In our setting it is only needed for the small projected matrix, where we evaluate $\varphi_k(T_m)e_1$ directly through the eigendecomposition — here is the check that both agree:

In [ ]:
Tm = T[:20, :20]                                            # from the Lanczos run above
aug = np.zeros((21, 21)); aug[:20, :20] = -dt*Tm; aug[:20, 20] = np.eye(20)[:, 0]
phi1_aug = sla.expm(aug)[:20, 20]
th, Q = sla.eigh(Tm); phi1_eig = Q @ (phi(1, -dt*th) * Q[0, :])
print(f"max difference between augmented-matrix and eigendecomposition phi_1(-dt T_m) e_1: {abs(phi1_aug - phi1_eig).max():.1e}")

## 7. Comparison on the fine mesh

Time per application of $e^{-\Delta t A}$ for $\Delta t = 0.01$ at a tolerance of about $10^{-8}$. The reference is shift-and-invert Lanczos with tolerance $10^{-13}$, cross-checked against the contour integral with $N = 32$. Setup costs (factorisations) are listed separately: they are paid once per run if $\Delta t$ (hence $\gamma$ and the poles) is fixed.

In [ ]:
dt = 1e-2; P = fine
M, v = P["M"], P["u0"]
nrm = lambda w: sqrt(InnerProduct(w, M*w))
t0 = walltime(); Eref = KrylovExp(P["M"], P["K"], dt, P["freedofs"], tol=1e-13, maxm=100); yref = (Eref*v).Evaluate(); print(f"reference: SI Lanczos m = {Eref.info['m']}, {walltime()-t0:.2f} s")
t0 = walltime(); C32 = ContourExp(P["Mc"], P["Kc"], dt, P["freedofsc"], N=32, M=P["M"]); ycont = (C32*v).Evaluate(); print(f"cross-check: contour N=32, {walltime()-t0:.2f} s,  rel. difference {nrm((yref-ycont).Evaluate())/nrm(yref):.1e}")

results = {}
def timeit(name, setup, apply):
    t0 = walltime(); obj = setup(); ts = walltime() - t0
    t0 = walltime(); res = apply(obj); ta = walltime() - t0
    results[name] = (ts, ta, nrm((res - yref).Evaluate())/nrm(yref))
    print(f"{name:32s} setup {ts:7.3f} s   apply {ta:7.3f} s   rel. error {results[name][2]:.1e}")

timeit("polynomial Lanczos (tol 1e-8)", lambda: KrylovExp(P["M"], P["K"], dt, P["freedofs"], poly=True, tol=1e-8, maxm=400), lambda E: (E*v).Evaluate())
timeit("shift-and-invert Lanczos (1e-8)", lambda: KrylovExp(P["M"], P["K"], dt, P["freedofs"], tol=1e-8), lambda E: (E*v).Evaluate())
Lf = lanczos_spectral_bound(P["applyA"], P["M"], v, m=40)
timeit(f"Chebyshev (degree {int(dt*Lf/2)+60})", lambda: None, lambda _: chebyshev_exp(P["applyA"], v, dt, Lf, int(dt*Lf/2)+60))
timeit("contour N=24 (12 complex solves)", lambda: ContourExp(P["Mc"], P["Kc"], dt, P["freedofsc"], N=24, M=P["M"]), lambda C: (C*v).Evaluate())
timeit("expm_multiply (Taylor)", lambda: None, lambda _: expm_multiply_FE(P, dt)[0])

In [ ]:
names = list(results); ts = [results[n][0] for n in names]; ta = [results[n][1] for n in names]
plt.figure(figsize=(7, 3.5)); yy = np.arange(len(names))
plt.barh(yy - 0.2, ta, 0.4, label="apply"); plt.barh(yy + 0.2, ts, 0.4, label="setup (factorisations)")
plt.yticks(yy, names); plt.xscale("log"); plt.xlabel("seconds"); plt.legend(); plt.grid(axis="x"); plt.title(f"fine mesh, ndof = {P['ndof']}, dt = {dt}"); plt.tight_layout(); plt.show()

## Summary

* **Polynomial methods** (Lanczos, Taylor/`expm_multiply`, Chebyshev) need only products with $A$ — each a mass matrix solve here — but their cost grows with $\Delta t\lambda_{\max} \sim \Delta t/h^2$: $\sqrt{\rho}$ iterations for Lanczos, $\rho$ for the other two. They are the methods of choice when no factorisation is affordable (very large 3D problems, GPUs) and $\Delta t$ is small.
* **Shift-and-invert Lanczos** costs one factorisation of $M + \gamma K$ (the implicit Euler matrix) and 10–30 back-substitutions per step, independent of $h$ and $\Delta t$; with an inner iterative solver it also works matrix-free.
* **Contour integrals / rational approximations** cost $N/2 \approx 12$ complex factorisations and as many back-substitutions per step, with no inner products and perfect parallelism; the same poles serve all $\varphi_k$ (with the recursion via $K^{-1}$).
* All methods evaluate $\varphi$-functions through the same machinery; use the eigendecomposition (or the augmented matrix) of the small projected matrix, never $(e^z - 1)/z$ in floating point for small $z$.

Not covered here: restarted Krylov methods (Eiermann–Ernst 2006; Afanasjew et al. 2008) to bound the basis size, Leja interpolation (Caliari et al.), extended and general rational Krylov spaces (Güttel 2013), and adaptive substepping as in `expokit`/`phipm`/`KIOPS`.

In [ ]:
# tests
assert results["shift-and-invert Lanczos (1e-8)"][2] < 1e-7
assert results["contour N=24 (12 complex solves)"][2] < 1e-9
assert results["polynomial Lanczos (tol 1e-8)"][2] < 1e-6
assert results["Chebyshev (degree %d)" % (int(dt*Lf/2)+60)][2] < 1e-8
assert results["expm_multiply (Taylor)"][2] < 1e-8
assert nrm((yref-ycont).Evaluate())/nrm(yref) < 1e-10
print("all tests passed")

### References
* A. H. Al-Mohy, N. J. Higham, *Computing the action of the matrix exponential, with an application to exponential integrators*, SIAM J. Sci. Comput. 33 (2011) 488–511. [doi:10.1137/100788860](https://doi.org/10.1137/100788860)
* L. Bergamaschi, M. Vianello, *Efficient computation of the exponential operator for large, sparse, symmetric matrices*, Numer. Linear Algebra Appl. 7 (2000) 27–45.
* V. Druskin, L. Knizhnerman, *Two polynomial methods of calculating functions of symmetric matrices*, USSR Comput. Math. Math. Phys. 29 (1989) 112–121.
* M. Eiermann, O. G. Ernst, *A restarted Krylov subspace method for the evaluation of matrix functions*, SIAM J. Numer. Anal. 44 (2006) 2481–2504. [doi:10.1137/050633846](https://doi.org/10.1137/050633846)
* J. van den Eshof, M. Hochbruck, *Preconditioning Lanczos approximations to the matrix exponential*, SIAM J. Sci. Comput. 27 (2006) 1438–1457. [doi:10.1137/040605461](https://doi.org/10.1137/040605461)
* S. Güttel, *Rational Krylov approximation of matrix functions: numerical methods and optimal pole selection*, GAMM-Mitt. 36 (2013) 8–31. [doi:10.1002/gamm.201310002](https://doi.org/10.1002/gamm.201310002)
* M. Hochbruck, C. Lubich, *On Krylov subspace approximations to the matrix exponential operator*, SIAM J. Numer. Anal. 34 (1997) 1911–1925. [doi:10.1137/S0036142995280572](https://doi.org/10.1137/S0036142995280572)
* Y. Saad, *Analysis of some Krylov subspace approximations to the matrix exponential operator*, SIAM J. Numer. Anal. 29 (1992) 209–228. [doi:10.1137/0729014](https://doi.org/10.1137/0729014)
* R. B. Sidje, *Expokit: a software package for computing matrix exponentials*, ACM Trans. Math. Softw. 24 (1998) 130–156. [doi:10.1145/285861.285868](https://doi.org/10.1145/285861.285868)
* L. N. Trefethen, J. A. C. Weideman, T. Schmelzer, *Talbot quadratures and rational approximations*, BIT 46 (2006) 653–670. [doi:10.1007/s10543-006-0077-9](https://doi.org/10.1007/s10543-006-0077-9)
* J. A. C. Weideman, L. N. Trefethen, *Parabolic and hyperbolic contours for computing the Bromwich integral*, Math. Comp. 76 (2007) 1341–1356. [doi:10.1090/S0025-5718-07-01945-X](https://doi.org/10.1090/S0025-5718-07-01945-X)